# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What kind of substances use trends are attributing to overdoses in Allegheny County?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-12 18:54:18 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-12T18:54:18.187291")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `overdose substance drug trends Allegheny County`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 0 datasets matching 'overdose substance drug trends Allegheny County'

```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'overdose substance drug trends Allegheny County', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `overdose substance drug trends Allegheny County`

**Result preview:**
```
Found 6 datasets matching 'overdose substance drug trends Allegheny County'

1. **Allegheny County Fatal Accidental Overdoses**
   ID: `allegheny-county-fatal-accidental-overdoses`
   Fatal accidental overdose incidents in Allegheny County, denoting age, gender, race, drugs present, zip code of incident and zip code of residence.  Zip code of incident is where
   - Fatal Accidental Overdoses (CSV) [DataStore] ID: `1c59b26a-1684-4bfb-92f7-205b947530cf`
   - Overdose Free PA Website (HTML) ID: `ce21c073-191d-488d-b544-9866cd041fe9`
   - Opiate-Related Overdose Deaths in Allegheny County: Risks
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'overdose substance drug trends Allegheny County', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Dataset Details

**Dataset:** `allegheny-county-fatal-accidental-overdoses`

**Result preview:**
```
# Allegheny County Fatal Accidental Overdoses

Fatal accidental overdose incidents in Allegheny County, denoting age, gender, race, drugs present, zip code of incident and zip code of residence.

Zip code of incident is where the Office of the Medical Examiner received the body, not necessarily where the overdose occurred. 

Data includes closed cases only and the previous calendar year data will be updated monthly until the close of the current calendar year. For example, the 2014 resource will be updated monthly until December 2015. 

**Pre-processing:** The cumulative Fatal Accidental
```


In [ ]:
# Step 3: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'allegheny-county-fatal-accidental-overdoses'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 4: Load Data from Resource

**Resource ID:** `1c59b26a-1684-4bfb-92f7-205b947530cf`
**Limit:** 5

**Result preview:**
```
Resource: 1c59b26a-1684-4bfb-92f7-205b947530cf
Total records: 7,678
Loaded: 5
Fields (19): death_date_and_time, manner_of_death, age, sex, race, case_dispo, combined_od1, combined_od2, combined_od3, combined_od4, combined_od5, combined_od6, combined_od7, combined_od8, combined_od9, combined_od10, incident_zip, decedent_zip, case_year

Sample (5 rows):

death_date_and_time manner_of_death  age sex race case_dispo         combined_od1 combined_od2 combined_od3 combined_od4 combined_od5 combined_od6 combined_od7 combined_od8 combined_od9 combined_od10 incident_zip decedent_zip  case_year
2025-06-
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '1c59b26a-1684-4bfb-92f7-205b947530cf', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT combined_od1, COUNT(*) as count FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" WHERE combined_od1 IS NOT NULL AND combined_od1 != 'None' GROUP BY combined_od1 ORDER BY count DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT combined_od1, COUNT(*) as count FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" WHERE combined_od1 IS NOT NULL AND combined_od1 != 'None' GROUP BY combined_od1 ORDER BY count DESC LIMIT 20
Rows: 20
Columns: combined_od1, count

           combined_od1  count
               Fentanyl   2226
                Cocaine   1726
                Alcohol   1069
                 Heroin    552
             Alprazolam    499
        Acetyl Fentanyl    226
             Clonazepam    111
              Methadone     92
               Morphine     79
               Diazepam     75
             Citalopram
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT combined_od1, COUNT(*) as count FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" WHERE combined_od1 IS NOT NULL AND combined_od1 != \'None\' GROUP BY combined_od1 ORDER BY count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT case_year, COUNT(*) as total_deaths FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" GROUP BY case_year ORDER BY case_year ASC
```

**Result preview:**
```
SQL: SELECT case_year, COUNT(*) as total_deaths FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" GROUP BY case_year ORDER BY case_year ASC
Rows: 20
Columns: case_year, total_deaths

 case_year  total_deaths
      2007           223
      2008           231
      2009           219
      2010           222
      2011           259
      2012           286
      2013           269
      2014           306
      2015           422
      2016           643
      2017           735
      2018           427
      2019           492
      2020           544
      2021           581
      2022           58
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT case_year, COUNT(*) as total_deaths FROM "1c59b26a-1684-4bfb-92f7-205b947530cf" GROUP BY case_year ORDER BY case_year ASC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql

SELECT case_year,
  SUM(CASE WHEN combined_od1 = 'Fentanyl' OR combined_od2 = 'Fentanyl' OR combined_od3 = 'Fentanyl' OR combined_od4 = 'Fentanyl' OR combined_od5 = 'Fentanyl' THEN 1 ELSE 0 END) as fentanyl,
  SUM(CASE WHEN combined_od1 = 'Heroin' OR combined_od2 = 'Heroin' OR combined_od3 = 'Heroin' OR combined_od4 = 'Heroin' OR combined_od5 = 'Heroin' THEN 1 ELSE 0 END) as heroin,
  SUM(CASE WHEN combined_od1 = 'Cocaine' OR combined_od2 = 'Cocaine' OR combined_od3 = 'Cocaine' OR combined_od4 = 'Cocaine' OR combined_od5 = 'Cocaine' THEN 1 ELSE 0 END) as cocaine,
  SUM(CASE WHEN combined_od1 = 'Alcohol' OR combined_od2 = 'Alcohol' OR combined_od3 = 'Alcohol' OR combined_od4 = 'Alcohol' OR combined_od5 = 'Alcohol' THEN 1 ELSE 0 END) as alcohol,
  SUM(CASE WHEN combined_od1 = 'Methamphetamine' OR combined_od2 = 'Methamphetamine' OR combined_od3 = 'Methamphetamine' OR combined_od4 = 'Methamphetamine' OR combined_od5 = 'Methamphetamine' THEN 1 ELSE 0 END) as methamphetamine,
  SUM(CASE WHEN combined_od1 ILIKE '%Fentanyl%' OR combined_od2 ILIKE '%Fentanyl%' OR combined_od3 ILIKE '%Fentanyl%' OR combined_od4 ILIKE '%Fentanyl%' OR combined_od5 ILIKE '%Fentanyl%' THEN 1 ELSE 0 END) as any_fentanyl,
  COUNT(*) as total
FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
GROUP BY case_year ORDER BY case_year ASC

```

**Result preview:**
```
SQL: 
SELECT case_year,
  SUM(CASE WHEN combined_od1 = 'Fentanyl' OR combined_od2 = 'Fentanyl' OR combined_od3 = 'Fentanyl' OR combined_od4 = 'Fentanyl' OR combined_od5 = 'Fentanyl' THEN 1 ELSE 0 END) as fentanyl,
  SUM(CASE WHEN combined_od1 = 'Heroin' OR combined_od2 = 'Heroin' OR combined_od3 = 'Heroin' OR combined_od4 = 'Heroin' OR combined_od5 = 'Heroin' THEN 1 ELSE 0 END) as heroin,
  SUM(CASE WHEN combined_od1 = 'Cocaine' OR combined_od2 = 'Cocaine' OR combined_od3 = 'Cocaine' OR combined_od4 = 'Cocaine' OR combined_od5 = 'Cocaine' THEN 1 ELSE 0 END) as cocaine,
  SUM(CASE WHEN combined
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT case_year,\n  SUM(CASE WHEN combined_od1 = \'Fentanyl\' OR combined_od2 = \'Fentanyl\' OR combined_od3 = \'Fentanyl\' OR combined_od4 = \'Fentanyl\' OR combined_od5 = \'Fentanyl\' THEN 1 ELSE 0 END) as fentanyl,\n  SUM(CASE WHEN combined_od1 = \'Heroin\' OR combined_od2 = \'Heroin\' OR combined_od3 = \'Heroin\' OR combined_od4 = \'Heroin\' OR combined_od5 = \'Heroin\' THEN 1 ELSE 0 END) as heroin,\n  SUM(CASE WHEN combined_od1 = \'Cocaine\' OR combined_od2 = \'Cocaine\' OR combined_od3 = \'Cocaine\' OR combined_od4 = \'Cocaine\' OR combined_od5 = \'Cocaine\' THEN 1 ELSE 0 END) as cocaine,\n  SUM(CASE WHEN combined_od1 = \'Alcohol\' OR combined_od2 = \'Alcohol\' OR combined_od3 = \'Alcohol\' OR combined_od4 = \'Alcohol\' OR combined_od5 = \'Alcohol\' THEN 1 ELSE 0 END) as alcohol,\n  SUM(CASE WHEN combined_od1 = \'Methamphetamine\' OR combined_od2 = \'Methamphetamine\' OR combined_od3 = \'Methamphetamine\' OR combined_od4 = \'Methamphetamine\' OR combined_od5 = \'Methamphetamine\' THEN 1 ELSE 0 END) as methamphetamine,\n  SUM(CASE WHEN combined_od1 ILIKE \'%Fentanyl%\' OR combined_od2 ILIKE \'%Fentanyl%\' OR combined_od3 ILIKE \'%Fentanyl%\' OR combined_od4 ILIKE \'%Fentanyl%\' OR combined_od5 ILIKE \'%Fentanyl%\' THEN 1 ELSE 0 END) as any_fentanyl,\n  COUNT(*) as total\nFROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\nGROUP BY case_year ORDER BY case_year ASC\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 8: SQL Analysis Query

**SQL:**
```sql

SELECT 
  COALESCE(combined_od1, '') as substance,
  COUNT(*) as count
FROM (
  SELECT combined_od1 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od2 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od3 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od4 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od5 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od6 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od7 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od8 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od9 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
  UNION ALL SELECT combined_od10 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"
) all_substances
WHERE combined_od1 IS NOT NULL AND combined_od1 != 'None' AND combined_od1 != ''
GROUP BY combined_od1
ORDER BY count DESC
LIMIT 20

```

**Result preview:**
```
SQL error (HTTP 403): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Authorization Error", "message": "Access denied: Not authorized to call function COALESCE"}, "success": false}
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  COALESCE(combined_od1, \'\') as substance,\n  COUNT(*) as count\nFROM (\n  SELECT combined_od1 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od2 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od3 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od4 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od5 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od6 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od7 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od8 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od9 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n  UNION ALL SELECT combined_od10 FROM "1c59b26a-1684-4bfb-92f7-205b947530cf"\n) all_substances\nWHERE combined_od1 IS NOT NULL AND combined_od1 != \'None\' AND combined_od1 != \'\'\nGROUP BY combined_od1\nORDER BY count DESC\nLIMIT 20\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here is a comprehensive analysis of substance use trends contributing to fatal overdoses in Allegheny County, drawn from the **Allegheny County Fatal Accidental Overdoses** dataset on the WPRDC portal (7,678 records from 2007–2026):

---

## 📊 Overall Trend: Fatal Overdose Deaths by Year

| Year | Total Deaths |
|------|-------------|
| 2007 | 223 |
| 2011 | 259 |
| 2014 | 306 |
| **2015** | **422** ⬆️ Sharp rise begins |
| **2016** | **643** |
| **2017** | **735** ← Peak year |
| 2018 | 427 |
| 2019 | 492 |
| 2020 | 544 |
| 2021 | 581 |
| 2022 | 580 |
| 2023 | 554 |
| 2024 | 355 |
| 2025* | 298 *(partial)* |

Deaths more than **tripled** from 2007 to the 2017 peak, and while they dipped in 2018, they remained persistently elevated.

---

## 💊 Top Substances Involved in Overdose Deaths (All Years Combined)

| Substance | Cases (Primary) |
|-----------|----------------|
| **Fentanyl** | 2,226 |
| **Cocaine** | 1,726 |
| **Alcohol** | 1,069 |
| **Heroin** | 552 |
| Alprazolam (Xanax) | 499 |
| Acetyl Fentanyl | 226 |
| Clonazepam | 111 |
| Methadone | 92 |
| Morphine | 79 |
| Diazepam (Valium) | 75 |
| Methamphetamine | 56 |
| Oxycodone | 66 |

---

## 📈 Key Substance Trends Over Time

### 🔵 Fentanyl — The #1 Killer
- Nearly **absent before 2014** (only 8–26 cases/year)
- **Exploded in 2016–2017** (401 → 537 cases)
- Remains the **dominant substance** in every year since 2016
- In 2022, fentanyl or a fentanyl analog appeared in **504 of 580 deaths** (~87%)

### 🟤 Heroin — Sharp Decline
- Was the leading opioid pre-2016 (**156 cases in 2014**, **307 in 2016**)
- Has **collapsed dramatically** — only **47 cases in 2023** and **17 in 2025**
- Largely **displaced by fentanyl** in the illicit drug supply

### 🟡 Cocaine — Surging in Recent Years
- Relatively flat 2007–2014 (~66–97/year)
- Rose significantly after 2019 — **330 cases in 2022** and **346 in 2023**
- Now rivals fentanyl as a leading substance — likely driven by **fentanyl-laced cocaine**

### 🔴 Methamphetamine — Slow but Steady Rise
- Virtually zero before 2017 (≤15 cases)
- Grew to **65 cases in 2023** — a newer but growing trend

### 🟠 Alcohol — Consistent Contributor
- A steady background factor (~43–162 cases/year)
- Peaked in 2017 alongside other substances (162 cases)

---

## 🔑 Summary Takeaways

1. **Fentanyl has become the dominant driver** of overdose deaths, replacing heroin almost entirely since 2016.
2. **Cocaine-involved deaths are surging**, likely due to contamination with fentanyl (polysubstance use).
3. **Heroin has largely disappeared** from the overdose landscape, replaced by synthetic opioids.
4. **Methamphetamine is a growing concern**, now appearing in ~65+ deaths/year.
5. **Benzodiazepines** (Xanax/Valium) and alcohol remain consistent co-factors in many deaths.
6. The drug crisis peaked in **2017 (735 deaths)**, improved briefly in 2018, then stabilized at a high plateau (~550–580/year) through 2022.

> **Source:** *Allegheny County Fatal Accidental Overdoses* dataset — Western PA Regional Data Center (WPRDC), provided by the Allegheny County Medical Examiner's Office. Note: 2025–2026 data is partial/pending case closures.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-12

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-12 18:54:18
- **Query**: What kind of substances use trends are attributing to overdoses in Allegheny County?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
